In [12]:
!git clone https://{token}@github.com/{organization}/{repo}

Cloning into 'ipekgpt'...
remote: Write access to repository not granted.
fatal: unable to access 'https://github.com/IpekYoluGPT/ipekgpt/': The requested URL returned error: 403


In [10]:
"""
RAG-Based Organization Chatbot System with Turkish Gemma Model
===============================================================
FIXED VERSION: Corrected response extraction and cleaning
"""

# ============================================================================
# STEP 1: INSTALL REQUIRED LIBRARIES (FIXED GPU INSTALLATION)
# ============================================================================

# Ensure torch is installed
!pip install -q torch

# FIXED: Install llama-cpp-python with pre-built cuBLAS wheel
print("⏳ Installing llama-cpp-python with GPU support (using pre-built wheel)...")

# First, uninstall any existing version
!pip uninstall -y llama-cpp-python 2>/dev/null

# Install the pre-built wheel with CUDA support
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

# Install other libraries
!pip install -q langchain langchain-community
!pip install -q chromadb sentence-transformers
!pip install -q huggingface-hub

print("✅ All libraries installed successfully!")

# =DEBUG: Verify PyTorch and CUDA
import torch
if torch.cuda.is_available():
    print(f"✅ PyTorch: Found GPU! ({torch.cuda.get_device_name(0)})")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("❌ PyTorch: COULD NOT FIND GPU. Check Runtime type.")

# Verify llama-cpp-python installation
try:
    import llama_cpp
    print(f"✅ llama-cpp-python installed successfully! (version: {llama_cpp.__version__})")
except ImportError as e:
    print(f"❌ Failed to import llama-cpp-python: {e}")

# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================

import os
import json
import re
from typing import List, Dict
from pathlib import Path

# LangChain imports
from langchain.docstore.document import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms.base import LLM

# Llama CPP
from llama_cpp import Llama

print("✅ Libraries imported successfully!")

# ============================================================================
# STEP 4: CONFIGURATION
# ============================================================================

class Config:
    """Configuration for the model"""

    # Embedding model (Turkish support)
    EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    # Vector database
    VECTOR_DB_PATH = "./chroma_db"
    COLLECTION_NAME = "org_knowledge_turkish"

    # Turkish Gemma Model Configuration
    GEMMA_REPO_ID = "ytu-ce-cosmos/Turkish-Gemma-9b-T1-GGUF"
    GEMMA_FILENAME = "*Q4_K.gguf"  # Q4_K is good balance of quality/speed

    # Gemma inference parameters
    GEMMA_PARAMS = {
        "n_gpu_layers": -1,       # Use -1 to offload all layers to GPU
        "n_threads": 4,           # Number of CPU threads
        # INCREASED: Context window (Input + History + Answer space)
        "n_ctx": 8192,  # Increased from 2048 to allow more documents + reasoning
        "n_predict": 2048, # Increased from 1024 to prevent cut-offs
        "top_k": 40,    # Slightly higher for creativity
        "top_p": 0.90,  # Slightly lower to keep it focused
        "temp": 0.1,    # !!! LOWER this! We want facts, not creative writing.
        "repeat_penalty": 1.1,
    }

    # Retrieval settings
    TOP_K_RESULTS = 10   # Number of relevant Q&A pairs to retrieve

config = Config()

# ============================================================================
# STEP 5: DATA LOADING (Pre-chunked Q&A Format)
# ============================================================================

class QADataLoader:
    """Loads pre-chunked Q&A data from various formats"""

    def __init__(self):
        self.stats = {
            "total_qa_pairs": 0,
            "by_category": {},
            "by_file": {}
        }

    def load_json_qa(self, file_path: str) -> List[Document]:
        """Load Q&A pairs from JSON format"""
        documents = []

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            qa_pairs = data.get('data', [])

            # Extract category from folder structure
            path_parts = Path(file_path).parts
            main_category = "Genel"
            sub_category = Path(file_path).stem

            # Get main category from folder (e.g., "01_TEMEL_BILGILER")
            for part in path_parts:
                if part.startswith(('01_', '02_', '03_', '04_', '05_')):
                    main_category = part.split('_', 1)[1] if '_' in part else part
                    break

            for item in qa_pairs:
                # Combine Q&A into a single text chunk
                content = f"Soru: {item['question']}\n\nCevap: {item['answer']}"

                # Convert lists to strings for ChromaDB compatibility
                keywords = item.get('keywords', [])
                if isinstance(keywords, list):
                    keywords_str = ", ".join(str(k) for k in keywords)
                else:
                    keywords_str = str(keywords)

                related_questions = item.get('related_questions', [])
                if isinstance(related_questions, list):
                    related_str = " | ".join(str(q) for q in related_questions)
                else:
                    related_str = str(related_questions)

                metadata = {
                    "question": str(item['question']),
                    "answer": str(item['answer']),
                    "main_category": str(main_category),
                    "sub_category": str(sub_category),
                    "category": str(item.get('category', main_category)),
                    "keywords": keywords_str,
                    "related_questions": related_str,
                    "priority": str(item.get('priority', 'medium')),
                    "source": str(file_path),
                    "file_name": str(Path(file_path).name)
                }

                documents.append(Document(page_content=content, metadata=metadata))

            # Update statistics
            self.stats["by_file"][Path(file_path).name] = len(documents)
            if main_category not in self.stats["by_category"]:
                self.stats["by_category"][main_category] = 0
            self.stats["by_category"][main_category] += len(documents)

            print(f"✅ Loaded {len(documents)} Q&A pairs from {Path(file_path).name}")
            return documents

        except Exception as e:
            print(f"❌ Error loading {file_path}: {str(e)}")
            import traceback
            traceback.print_exc()
            return []

    def load_text_qa(self, file_path: str) -> List[Document]:
        """Load Q&A pairs from text format (Q: ... A: ... separated by ---)"""
        documents = []

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # Extract category from folder structure
            path_parts = Path(file_path).parts
            main_category = "Genel"

            for part in path_parts:
                if part.startswith(('01_', '02_', '03_', '04_', '05_')):
                    main_category = part.split('_', 1)[1] if '_' in part else part
                    break

            # Split by separator
            qa_blocks = content.split('---')

            for block in qa_blocks:
                block = block.strip()
                if not block:
                    continue

                # Extract Q, A, and optional fields
                lines = block.split('\n')
                question = ""
                answer = ""
                category = main_category
                keywords = ""
                priority = "medium"

                for line in lines:
                    line = line.strip()
                    if line.startswith('Q:'):
                        question = line[2:].strip()
                    elif line.startswith('A:'):
                        answer = line[2:].strip()
                    elif line.startswith('Category:'):
                        category = line[9:].strip()
                    elif line.startswith('Keywords:'):
                        keywords = line[9:].strip()
                    elif line.startswith('Priority:'):
                        priority = line[9:].strip()

                if question and answer:
                    content = f"Soru: {question}\n\nCevap: {answer}"
                    metadata = {
                        "question": str(question),
                        "answer": str(answer),
                        "main_category": str(main_category),
                        "sub_category": str(Path(file_path).stem),
                        "category": str(category),
                        "keywords": str(keywords),
                        "priority": str(priority),
                        "source": str(file_path),
                        "file_name": str(Path(file_path).name)
                    }
                    documents.append(Document(page_content=content, metadata=metadata))

            # Update statistics
            self.stats["by_file"][Path(file_path).name] = len(documents)
            if main_category not in self.stats["by_category"]:
                self.stats["by_category"][main_category] = 0
            self.stats["by_category"][main_category] += len(documents)

            print(f"✅ Loaded {len(documents)} Q&A pairs from {Path(file_path).name}")
            return documents

        except Exception as e:
            print(f"❌ Error loading {file_path}: {str(e)}")
            return []

    def load_from_directory(self, dir_path: str) -> List[Document]:
        """Load all Q&A files from a directory (supports hierarchical structure)"""
        all_docs = []

        # Recursively find all JSON and TXT files
        for file_path in sorted(Path(dir_path).rglob('*')):
            if file_path.suffix.lower() == '.json':
                docs = self.load_json_qa(str(file_path))
                all_docs.extend(docs)
            elif file_path.suffix.lower() == '.txt':
                docs = self.load_text_qa(str(file_path))
                all_docs.extend(docs)

        self.stats["total_qa_pairs"] = len(all_docs)

        print(f"\n{'='*70}")
        print(f"✅ TOPLAM {len(all_docs)} Q&A ÇİFTİ YÜKLENDİ")
        print(f"{'='*70}")

        return all_docs

    def print_statistics(self):
        """Print detailed loading statistics"""
        print("\n" + "="*70)
        print("📊 VERİ YÜKLEME İSTATİSTİKLERİ")
        print("="*70)

        print(f"\n🔢 Toplam Q&A Çifti: {self.stats['total_qa_pairs']}")

        print("\n📁 Kategorilere Göre Dağılım:")
        for category, count in sorted(self.stats['by_category'].items()):
            percentage = (count / self.stats['total_qa_pairs'] * 100) if self.stats['total_qa_pairs'] > 0 else 0
            print(f"  • {category}: {count} çift (%{percentage:.1f})")

        print("\n📄 Dosyalara Göre Dağılım:")
        for file_name, count in sorted(self.stats['by_file'].items()):
            print(f"  • {file_name}: {count} çift")

        print("="*70 + "\n")

# ============================================================================
# STEP 6: VECTOR STORE
# ============================================================================

class VectorStore:
    """Manages the vector database"""

    def __init__(self, embedding_model_name: str, db_path: str, collection_name: str):
        print("⏳ Initializing Turkish embedding model...")

        # Check for CUDA availability
        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"  (Using device: {device} for embeddings)")

        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model_name,
            model_kwargs={'device': device}
        )
        self.db_path = db_path
        self.collection_name = collection_name
        self.vectorstore = None
        print("✅ Embedding model ready!")

    def create_from_documents(self, documents: List[Document]):
        """Create a new vector store from Q&A documents"""
        print(f"⏳ Creating vector embeddings for {len(documents)} Q&A pairs...")

        # FIXED: Ensure directory exists and has proper permissions
        import os
        import shutil

        # Remove old database if it exists
        if os.path.exists(self.db_path):
            try:
                shutil.rmtree(self.db_path)
                print(f"  Removed old database at {self.db_path}")
            except Exception as e:
                print(f"  Warning: Could not remove old database: {e}")

        # Create fresh directory
        os.makedirs(self.db_path, exist_ok=True)

        # Use PersistentClient instead of default Client
        import chromadb
        from chromadb.config import Settings

        # Create client with explicit settings
        chroma_client = chromadb.PersistentClient(
            path=self.db_path,
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )

        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            client=chroma_client,
            collection_name=self.collection_name
        )

        print("✅ Vector store created and persisted!")
        return self.vectorstore

    def load_existing(self):
        """Load an existing vector store"""
        print("⏳ Loading existing vector store...")

        import chromadb
        from chromadb.config import Settings

        # Create client with explicit settings
        chroma_client = chromadb.PersistentClient(
            path=self.db_path,
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )

        self.vectorstore = Chroma(
            client=chroma_client,
            embedding_function=self.embeddings,
            collection_name=self.collection_name
        )

        print("✅ Vector store loaded!")
        return self.vectorstore

# ============================================================================
# STEP 7: TURKISH GEMMA LLM WRAPPER (FIXED)
# ============================================================================

class TurkishGemmaLLM(LLM):
    """Custom LLM wrapper for Turkish Gemma model"""

    model: Llama = None

    def __init__(self, repo_id: str, filename: str, **kwargs):
        super().__init__()
        print("⏳ Downloading and loading Turkish Gemma model...")
        print("   (This may take several minutes on first run)")

        self.model = Llama.from_pretrained(
            repo_id=repo_id,
            filename=filename,
            verbose=False,
            **config.GEMMA_PARAMS
        )

        print("✅ Turkish Gemma model loaded!")

        # Check if GPU is being used
        if config.GEMMA_PARAMS.get('n_gpu_layers', 0) > 0 or config.GEMMA_PARAMS.get('n_gpu_layers') == -1:
            print("   🚀 Model is using GPU acceleration!")

    @property
    def _llm_type(self) -> str:
        return "turkish_gemma"

    def _call(self, prompt: str, stop=None) -> str:
        """Generate response from the model - FIXED VERSION"""

        # Format prompt with Gemma's special tokens
        formatted_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"

        # Generate response
        response = self.model(
            formatted_prompt,
            stop=["<end_of_turn>", "</s>"],  # Stop tokens
            max_tokens=config.GEMMA_PARAMS['n_predict'],
            temperature=config.GEMMA_PARAMS['temp'],
            top_p=config.GEMMA_PARAMS['top_p'],
            top_k=config.GEMMA_PARAMS['top_k'],
            repeat_penalty=config.GEMMA_PARAMS['repeat_penalty']
        )

        # Extract raw text
        raw_text = response['choices'][0]['text']

        if "<think>" in raw_text and "</think>" not in raw_text:
            print("[WARNING] Model output cut off during thinking process. Increasing n_predict might help.")
            # We strip everything, because there is no answer yet.
            return "Üzgünüm, düşünme süreci yarıda kesildiği için cevap oluşturulamadı."

        # Debug: Print raw response
        print(f"\n[DEBUG] Raw model output: {raw_text[:200]}...")

        # FIXED: More careful cleaning
        # 1. Remove <think> tags and their content
        cleaned_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL)

        # 2. Remove other XML-like tags but keep the content
        cleaned_text = re.sub(r'<[^>]+>', '', cleaned_text)

        # 3. Remove citation markers like [+]
        cleaned_text = re.sub(r'\[\+\]', '', cleaned_text)

        # 4. Clean up excessive whitespace but preserve single newlines
        cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
        cleaned_text = re.sub(r' +', ' ', cleaned_text)

        # --- 2. REMOVE META-TALK ("Bağlamdaki bilgilere göre...") ---
        # List of phrases local models love to use when doing RAG
        meta_phrases = [
            r"Bağlamdaki bilgilere göre,?",
            r"Verilen metne göre,?",
            r"Sağlanan dokümanlara dayanarak,?",
            r"Bağlamda belirtildiği üzere,?",
            r"Dokümanlara göre,?"
        ]

        for phrase in meta_phrases:
            # Case insensitive replacement
            cleaned_text = re.sub(phrase, "", cleaned_text, flags=re.IGNORECASE)
        # 5. Strip leading/trailing whitespace
        cleaned_text = cleaned_text.strip()

        if cleaned_text:
            cleaned_text = cleaned_text[0].upper() + cleaned_text[1:]

        # If we got nothing, return a fallback message
        if not cleaned_text or len(cleaned_text) < 10:
            print("[WARNING] Model output was empty or too short after cleaning!")
            return "Üzgünüm, bu soruya şu anda net bir cevap veremiyorum. Lütfen soruyu farklı şekilde sormayı deneyin."

        return cleaned_text

# ============================================================================
# STEP 8: RAG PIPELINE (IMPROVED)
# ============================================================================

class TurkishRAGChatbot:
    """RAG chatbot using Turkish Gemma model"""

    def __init__(self, vectorstore):
        """Initialize the RAG chatbot with Turkish Gemma"""

        # Initialize Turkish Gemma LLM
        self.llm = TurkishGemmaLLM(
            repo_id=config.GEMMA_REPO_ID,
            filename=config.GEMMA_FILENAME
        )

        # Improved prompt template
        self.prompt_template = """Sen İpekyolu Girişimci Kuluçka Merkezimiz için yardımcı bir asistansın.
Aşağıdaki bağlam bilgilerini kullanarak soruyu doğru ve detaylı şekilde cevapla.

KURALLAR:
1. Cevabını SADECE verilen bağlamdaki bilgilere dayandır
2. Net, anlaşılır ve dostça bir dil kullan
3. Bağlamda cevap yoksa "Bu konuda bilgi tabanımda yeterli bilgi bulunmuyor" de
4. Kesinlikle bilgi uydurma veya tahmin yapma
5. Kesinlikle cevap'ta bağlam kelimesinden vs. bahsetme.
6. Direkt ol, yani direkt cevabı söyle veya açıkla.

Bağlam:
{context}

Soru: {question}

Cevap:"""

        self.PROMPT = PromptTemplate(
            template=self.prompt_template,
            input_variables=["context", "question"]
        )

        # Use MMR retriever to avoid duplicates
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(
                search_type="mmr",  # Maximum Marginal Relevance
                search_kwargs={
                    'k': config.TOP_K_RESULTS,
                    'fetch_k': 10
                }
            ),
            return_source_documents=True,
            chain_type_kwargs={"prompt": self.PROMPT}
        )

        print("✅ Turkish RAG Chatbot initialized and ready!")

    def ask(self, question: str, show_sources: bool = True) -> Dict:
        """Ask a question and get an answer

        Args:
            question: The question to ask
            show_sources: If True, display detailed source information (default: True)
        """
        print(f"\n{'='*70}")
        print(f"❓ Soru: {question}")
        print(f"{'='*70}")

        try:
            result = self.qa_chain.invoke({"query": question})

            answer = result['result']
            sources = result['source_documents']

            print(f"\n💡 Cevap:\n{answer}")
            print(f"\n{'='*70}\n")

            if show_sources and sources:
                print(f"\n📚 Bilgi tabanından {len(sources)} kaynak kullanıldı:\n")
                for i, doc in enumerate(sources, 1):
                    main_cat = doc.metadata.get('main_category', 'N/A')
                    sub_cat = doc.metadata.get('sub_category', 'N/A')
                    priority = doc.metadata.get('priority', 'N/A')
                    print(f"  Kaynak {i}: [{main_cat} > {sub_cat}] (Öncelik: {priority})")
                    question_preview = doc.metadata.get('question', 'N/A')
                    if len(question_preview) > 80:
                        question_preview = question_preview[:80] + "..."
                    print(f"    Soru: {question_preview}")

                    keywords = doc.metadata.get('keywords', '')
                    if keywords:
                        print(f"    Anahtar Kelimeler: {keywords}")
                    print()

            return {
                "answer": answer,
                "sources": sources,
                "num_sources": len(sources),
                "categories_used": [doc.metadata.get('main_category') for doc in sources]
            }

        except Exception as e:
            print(f"\n❌ Hata oluştu: {str(e)}")
            import traceback
            traceback.print_exc()
            return {
                "answer": "Bir hata oluştu. Lütfen tekrar deneyin.",
                "sources": [],
                "num_sources": 0,
                "categories_used": []
            }

# ============================================================================
# STEP 9: MAIN WORKFLOW
# ============================================================================

def build_turkish_rag_system(qa_data_path: str):
    """
    Build the Turkish RAG system with pre-chunked Q&A data

    Args:
        qa_data_path: Path to IPEKYOLU_RAG_VERISETI directory or a single Q&A file
    """

    print("=" * 70)
    print("🚀 İPEKYOLU RAG CHATBOT SİSTEMİ KURULUMU")
    print("=" * 70)

    # Step 1: Load pre-chunked Q&A data
    print("\n📂 ADIM 1: Q&A verisi yükleniyor...")

    loader = QADataLoader()
    documents = []

    if Path(qa_data_path).is_dir():
        documents = loader.load_from_directory(qa_data_path)
    elif qa_data_path.endswith('.json'):
        documents = loader.load_json_qa(qa_data_path)
    elif qa_data_path.endswith('.txt'):
        documents = loader.load_text_qa(qa_data_path)
    else:
        print("❌ Desteklenmeyen dosya formatı! JSON veya TXT kullanın.")
        return None

    if not documents:
        print("❌ Hiç Q&A verisi yüklenemedi!")
        return None

    if not Path(qa_data_path).is_dir():
        loader.stats["total_qa_pairs"] = len(documents)

    # Show detailed statistics
    loader.print_statistics()

    # Step 2: Create vector store
    print("\n🧠 ADIM 2: Vektör veritabanı oluşturuluyor...")
    vector_store = VectorStore(
        embedding_model_name=config.EMBEDDING_MODEL,
        db_path=config.VECTOR_DB_PATH,
        collection_name=config.COLLECTION_NAME
    )
    vectorstore = vector_store.create_from_documents(documents)

    # Step 3: Initialize chatbot
    print("\n🤖 ADIM 3: Turkish Gemma RAG chatbot başlatılıyor...")
    chatbot = TurkishRAGChatbot(vectorstore)

    print("\n" + "=" * 70)
    print("✅ İPEKYOLU RAG SİSTEMİ HAZIR!")
    print("=" * 70)

    return chatbot

# ============================================================================
# EK FONKSİYONLAR - YARDIMCI ARAÇLAR
# ============================================================================

def batch_test_questions(chatbot, questions: List[str]):
    """
    Birden fazla soruyu test et ve sonuçları karşılaştır

    Args:
        chatbot: RAG chatbot instance
        questions: Test edilecek sorular listesi
    """
    print("\n" + "="*70)
    print("🧪 TOPLU TEST BAŞLATILIYOR")
    print("="*70)

    results = []

    for i, question in enumerate(questions, 1):
        print(f"\n--- Test {i}/{len(questions)} ---")
        result = chatbot.ask(question, show_sources=False)
        results.append({
            "question": question,
            "answer": result["answer"],
            "num_sources": result["num_sources"],
            "categories": result["categories_used"]
        })
        print("-" * 70)

    # Summary
    print("\n" + "="*70)
    print("📊 TEST SONUÇ ÖZETİ")
    print("="*70)

    for i, res in enumerate(results, 1):
        print(f"\n{i}. {res['question'][:60]}...")
        print(f"   Cevap uzunluğu: {len(res['answer'])} karakter")
        print(f"   Kaynak sayısı: {res['num_sources']}")
        print(f"   Kategoriler: {set(res['categories'])}")

    return results


print("\n" + "=" * 70)
print("📖 İPEKYOLU RAG MODÜLÜ YÜKLENDİ - Kullanıma hazır!")
print("=" * 70)

# ============================================================================
# USAGE HELPER
# ============================================================================

def quick_start():
    """Quick start function for easy setup"""
    print("\n🚀 HIZLI BAŞLATMA:")
    print("1. Chatbot'u oluşturmak için:")
    print("   chatbot = build_turkish_rag_system('./IPEKYOLU_RAG_VERISETI')")
    print("\n2. Soru sormak için:")
    print("   chatbot.ask('Kurucusu kimdir?')")
    print("\n3. Birden fazla soru test etmek için:")
    print("   questions = ['Soru 1', 'Soru 2', 'Soru 3']")
    print("   batch_test_questions(chatbot, questions)")
    print("\n" + "=" * 70)

⏳ Installing llama-cpp-python with GPU support (using pre-built wheel)...
Found existing installation: llama_cpp_python 0.3.16
Uninstalling llama_cpp_python-0.3.16:
  Successfully uninstalled llama_cpp_python-0.3.16
✅ All libraries installed successfully!
✅ PyTorch: Found GPU! (Tesla T4)
   CUDA Version: 12.6
✅ llama-cpp-python installed successfully! (version: 0.3.16)
✅ Libraries imported successfully!

📖 İPEKYOLU RAG MODÜLÜ YÜKLENDİ - Kullanıma hazır!


In [11]:
# Clean restart - no manual directory creation needed
qa_data_path = './IPEKYOLU_RAG_VERISETI'
chatbot = build_turkish_rag_system(qa_data_path)

# Test it
if chatbot:
    response = chatbot.ask("ipek yolunda kaç otel kaç oda kaç kişilik konaklama var?")

🚀 İPEKYOLU RAG CHATBOT SİSTEMİ KURULUMU

📂 ADIM 1: Q&A verisi yükleniyor...
✅ Loaded 20 Q&A pairs from AnaSayfa.json
✅ Loaded 28 Q&A pairs from ArgeCalismalarimiz.json
✅ Loaded 17 Q&A pairs from BasindaBiz.json
✅ Loaded 13 Q&A pairs from KariyerVeEtkinliklerimiz.json
✅ Loaded 24 Q&A pairs from Linkedin.json
✅ Loaded 15 Q&A pairs from atolyelerimiz.json
✅ Loaded 27 Q&A pairs from basarilarimiz.json
✅ Loaded 18 Q&A pairs from hakkimizda.json
✅ Loaded 12 Q&A pairs from iletisim.json
✅ Loaded 16 Q&A pairs from sss.json

✅ TOPLAM 190 Q&A ÇİFTİ YÜKLENDİ

📊 VERİ YÜKLEME İSTATİSTİKLERİ

🔢 Toplam Q&A Çifti: 190

📁 Kategorilere Göre Dağılım:
  • Genel: 190 çift (%100.0)

📄 Dosyalara Göre Dağılım:
  • AnaSayfa.json: 20 çift
  • ArgeCalismalarimiz.json: 28 çift
  • BasindaBiz.json: 17 çift
  • KariyerVeEtkinliklerimiz.json: 13 çift
  • Linkedin.json: 24 çift
  • atolyelerimiz.json: 15 çift
  • basarilarimiz.json: 27 çift
  • hakkimizda.json: 18 çift
  • iletisim.json: 12 çift
  • sss.json: 16 çift

InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [ ]:
questions= ["makebot yapıyor musunuz?", "İpekyolu eğitimleri ücretli mi?" , "Eğitimlere nasıl katılabilirim", "Nasıl gönüllü olabilirim", "kaç metrekare üzerine kurulu?", "Bu organizasyonda ne gibi olanaklar var", "Girişimci kuluçka ne yapar", "2023'teki yarışmaya katıldığınız roket specsleri nelerdir?", "Hangi teknofest yarışmalarına katıldınız?"]
batch_test_questions(chatbot, questions)